# POC 6: Build a Self-Help Knowledge Base with RAG

**Pain point:** Your team's runbooks, FAQs, and IT procedures live in 5 different places — Confluence, email threads, a shared drive, someone's head. When a new joiner asks 'how do I reset my VPN access?', they get a link to a 40-page doc or a shrug.

**What this notebook shows:** A minimal RAG (Retrieval-Augmented Generation) knowledge base you can seed with your own docs. It retrieves only the relevant excerpt per query — so the LLM answers from *your* procedures, not generic internet advice. When the KB doesn't have an answer, it says so — no hallucinated steps.

**Three things you'll see:**
1. Grounded vs. ungrounded: same question, very different answers
2. Runtime knowledge add: new doc → immediately queryable, no retraining
3. Graceful not-found: KB misses → predictable fallback, not a confident hallucination

**You need:** A free Groq API key from https://console.groq.com

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

In [ ]:
# ℹ️ This installation step can time sometime
!pip install groq chromadb sentence-transformers -q


In [ ]:
import os
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY') or input('Enter Groq API key: ')

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL = 'qwen/qwen3.8-27b'

def call_llm(system_prompt, user_message, temperature=0.2, max_tokens=600):
    response = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print(f'LLM ready: {MODEL}')

## ⚠️ In-Memory KB — What This Means

This notebook uses an **in-memory knowledge base**. Everything stored here disappears when the Colab session ends.

That's fine for this POC — but if you want a KB that actually grows and survives, **one line is all that changes**:

```python
# This notebook — in-memory, session only
chroma_client = chromadb.EphemeralClient()

# Local disk — survives restarts
chroma_client = chromadb.PersistentClient(path="./my_kb")

# Google Drive from Colab — survives across sessions
from google.colab import drive
drive.mount('/content/drive')
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/my_kb")
```


Only the client init changes. The `add_to_kb()` and `retrieve()` functions you'll write below work identically across all of them.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# In-memory KB — swap to PersistentClient to make it survive restarts (see note above)
chroma_client = chromadb.EphemeralClient()
collection = chroma_client.create_collection(name='it_helpdesk')

# Free embedding model — runs locally in Colab, no API key needed, ~80MB
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

def add_to_kb(doc_id, text, metadata=None):
    """Embed a document and store it in the KB."""
    embedding = embed_model.encode(text).tolist()
    collection.add(
        documents=[text],
        embeddings=[embedding],
        ids=[doc_id],
        metadatas=[metadata or {}]
    )

def retrieve(query, n_results=2):
    """Find the top-n most relevant KB excerpts for a query."""
    query_embedding = embed_model.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    excerpts = results['documents'][0]
    return '\n\n---\n\n'.join(excerpts)

print('KB initialised (in-memory)')
print('Embedding model: all-MiniLM-L6-v2')

In [ ]:
# --- Seed the Knowledge Base ---
# These represent your team's IT procedures. Replace with your own docs to build your personal KB.

KB_DOCS = {
    'vpn-reset': """
VPN Access Reset — Nexus Corp IT Procedure

VPN client: GlobalProtect. Portal: vpn.nexus-corp.internal

To reset VPN access:
1. Raise an IT Service ticket at help.nexus-corp.internal
   Category: Network Access > VPN Reset
   Include: your employee ID and a manager approval email.
2. IT will revoke the existing credential and issue a new one within 2 business days.
3. New credentials will be sent to your corporate email.
4. Re-install GlobalProtect if prompted: help.nexus-corp.internal/software

Urgent resets (travel, client site): call IT Helpdesk on extension 4400 for same-day service.
""",

    'password-reset': """
Password Reset — Nexus Corp IT Procedure

Self-service (if enrolled in SSPR):
  Go to password.nexus-corp.internal and follow the prompts.
  Requires a registered phone number or backup email on file.

If not enrolled in SSPR or locked out:
  Call IT Helpdesk: extension 4400 (Mon-Fri, 08:00-18:00)
  Out of hours: email ithelpdesk@nexus-corp.internal — response next business day.

Lockout policy: 5 failed attempts triggers a 30-minute auto-lockout.
After 30 minutes, try again. If still locked, call x4400.

Password policy: min 12 characters, 1 uppercase, 1 number, 1 special character. Expires every 90 days.
""",

    'laptop-setup': """
New Laptop Setup Checklist — Nexus Corp

1. Wi-Fi: Connect to NexusCorp-Corporate (802.1x — use your AD credentials when prompted).
   Guest Wi-Fi SSID: NexusCorp-Guest (see Guest WiFi doc for daily password).

2. VPN: Install GlobalProtect from help.nexus-corp.internal/software.
   Portal address: vpn.nexus-corp.internal

3. SSO: Log in at myapps.nexus-corp.internal with your Active Directory credentials.

4. Disk encryption: BitLocker is pre-enabled on all corporate laptops.
   Escrow your recovery key: IT Portal > My Devices > Escrow BitLocker Key.

5. Required software: ServiceNow (tickets), Zoom, Microsoft 365, GlobalProtect.
   All available at help.nexus-corp.internal/software.

6. Raise a setup ticket if anything is missing or not working: help.nexus-corp.internal
""",

    'incident-ticket': """
How to Submit an IT Incident Ticket — Nexus Corp

Portal: help.nexus-corp.internal — select New Incident

Required fields:
  Summary: one-line description of the issue
  Priority: P1 / P2 / P3 / P4
  Business impact: describe what is currently blocked
  Affected users: number of users impacted
  Attachments: screenshots or error logs where available

SLA response times:
  P1 (service fully down): 1-hour initial response, 4-hour resolution target
  P2 (degraded service): 4-hour response, next business day resolution
  P3 (issue with workaround): 5 business days
  P4 (enhancement or request): 10 business days

For P1 incidents: also call x4400 immediately — do not wait for a ticket response.
""",

    'guest-wifi': """
Guest Wi-Fi Access — Nexus Corp

SSID: NexusCorp-Guest
Password: rotates daily. Check the lobby TV screen (reception level) or ask the reception desk.

Guest portal: guests.nexus-corp.internal
  Enter your name and a valid email address.
  A one-time access code is sent to your email.
  Session duration: 24 hours per registration.
  Bandwidth: 10 Mbps (traffic-shaped, streaming sites blocked).

Note: Guest Wi-Fi is on a separate VLAN. Guests cannot reach internal systems or printers.
For internal access, ask your host to raise a temporary visitor access request.
"""
}

for doc_id, text in KB_DOCS.items():
    add_to_kb(doc_id, text, metadata={'source': doc_id})

print(f'KB seeded with {len(KB_DOCS)} documents.')
print('Topics:', ', '.join(KB_DOCS.keys()))

In [ ]:
# --- UNGROUNDED: Ask without KB context ---
# The LLM has no knowledge of Nexus Corp's specific procedures, portal URLs, or SLAs.

QUERY = 'How do I reset my VPN access?'

ungrounded_system = 'You are a helpful IT support assistant.'

print(f'Query: "{QUERY}"')
print('=== UNGROUNDED OUTPUT ===')
print(call_llm(ungrounded_system, QUERY))

In [ ]:
# --- GROUNDED: Retrieve relevant KB excerpts, then answer ---
# Same question. The retriever finds the VPN doc and injects it as context.

GROUNDED_SYSTEM = """You are a self-help IT assistant for Nexus Corp.
Answer using ONLY the knowledge base excerpts provided.
If the answer is not in the excerpts, say exactly:
  "I don't have this in my knowledge base. Consider adding it — or contact the IT helpdesk."
Do not use outside knowledge. Do not invent procedures, URLs, or phone numbers."""

# Step 1: retrieve — semantic search finds the most relevant docs
context = retrieve(QUERY, n_results=2)
word_count = len(context.split())
print(f'Retrieved ~{word_count} words (~{int(word_count * 1.3)} tokens) — not the full KB')
print()

# Step 2: generate — LLM answers from context only
grounded_query = f'Knowledge base excerpts:\n\n{context}\n\nQuestion: {QUERY}'

print(f'Query: "{QUERY}"')
print('=== GROUNDED OUTPUT ===')
print(call_llm(GROUNDED_SYSTEM, grounded_query))

In [ ]:
# --- ADD A NEW DOCUMENT TO THE KB AT RUNTIME ---
# Simulates a new IT policy issued after the KB was originally seeded.
# No retraining. No redeployment. One function call.

NEW_DOC = """
Two-Factor Authentication (2FA) Setup — Microsoft Authenticator
Company policy effective: 1 October 2025. All employees must enrol by 15 September 2025.

Setup steps:
1. Download Microsoft Authenticator from the App Store (iOS) or Google Play (Android).
2. On your laptop, go to: myapps.nexus-corp.internal
3. Click your profile icon (top right) > Security Info > Add sign-in method.
4. Select 'Authenticator app' and click Add.
5. On your phone, open the Authenticator app > tap '+' > Work or school account > Scan QR code.
6. Scan the QR code shown on your laptop screen.
7. Approve the test notification on your phone to confirm setup is complete.

Enrolment deadline: 15 September 2025.
After 1 October, accounts without 2FA will be unable to log in without IT intervention.
Help: ithelpdesk@nexus-corp.internal or extension 4400.
"""

# ℹ️ We are upbating our KBA with new info
add_to_kb('2fa-setup', NEW_DOC, metadata={'source': '2fa-setup', 'added': 'runtime'})

print('New document added to KB: 2fa-setup')
print(f'KB now contains {collection.count()} documents (was 5).')

In [ ]:
# --- RE-QUERY: The KB now has the 2FA doc ---
# Same pipeline — retriever finds the just-added document.

NEW_QUERY = 'How do I set up two-factor authentication on my account?'

context = retrieve(NEW_QUERY, n_results=2)
grounded_query = f'Knowledge base excerpts:\n\n{context}\n\nQuestion: {NEW_QUERY}'

print(f'Query: "{NEW_QUERY}"')
print('=== OUTPUT (from just-added doc) ===')
print(call_llm(GROUNDED_SYSTEM, grounded_query))

In [ ]:
# --- GRACEFUL NOT-FOUND: KB has no matching answer ---
# The retriever will return something (it always does — it finds the closest match).
# But the LLM's system prompt says: if the excerpts don't answer the question, say so.

UNKNOWN_QUERY = 'How do I request a software licence for Figma?'

context = retrieve(UNKNOWN_QUERY, n_results=2)
grounded_query = f'Knowledge base excerpts:\n\n{context}\n\nQuestion: {UNKNOWN_QUERY}'

print(f'Query: "{UNKNOWN_QUERY}"')
print('=== OUTPUT (not in KB) ===')
print(call_llm(GROUNDED_SYSTEM, grounded_query))

## What just happened

**Three interactions, three patterns:**

**1. Ungrounded vs. grounded (VPN reset)**
The ungrounded model gives generic GlobalProtect advice from its training data — possibly correct for some organisation, certainly not for Nexus Corp. The grounded model gives specific portal URLs, the x4400 extension, the 2-business-day SLA, and the manager approval requirement — all from the KB, none invented.

**2. Runtime add (2FA policy)**
The KB was seeded without a 2FA doc. When the policy was issued, one `add_to_kb()` call made it immediately queryable. No retraining. No redeployment. The embedder runs at write time; retrieval is fast because the index already has the vector.

**3. Graceful not-found (Figma licence)**
The retriever always returns *something* — it finds the closest semantic match in the KB regardless. The system prompt is what enforces the boundary: *'if the excerpts don't answer the question, say so.'* This is the defining property of a grounded KB: predictable, auditable failure is better than a confident hallucination.

**Token cost:**
Each query injected ~2 KB excerpts, roughly 250–400 tokens of context. The full KB (6 docs) is ~900 tokens. Without retrieval, you'd inject all 900 tokens per query. With retrieval, you inject only what's semantically relevant — that's [Scenario 2 from the grounding series](https://sriharshacr.github.io/blogs/does-grounding-reduce-token-usage/): selective injection reduces per-call input tokens while maintaining accuracy.

---

**Where to take this next:**

| Improvement | What to do |
|---|---|
| **Persistence** | Swap `EphemeralClient()` → `PersistentClient(path='./my_kb')` — KB survives restarts |
| **Chunking** | Split long docs into ~300-token chunks before embedding — semantic search works better on chunks than full docs |
| **Metadata filtering** | Use ChromaDB `where` clauses to restrict search to a category (e.g. only 'network' docs for VPN questions) |
| **Self-updating** | After a helpful LLM response, offer to save it back into the KB — the system teaches itself over time |
| **Multi-format ingestion** | Parse PDFs, Confluence pages, or Markdown files and `add_to_kb()` each chunk |
